<a href="https://colab.research.google.com/github/GabzBarbosa/STOCK-CRAWLER-GOOGLE-SHEETS/blob/main/STOCK_CRAWLER.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [8]:
# ==============================================================================
# NETSHOES STOCK CRAWLER + GOOGLE SHEETS (AUTENTICAÇÃO NATIVA COLAB)
# ==============================================================================
#
#
# ESTRUTURA DA PLANILHA ESPERADA:
# COLUNA A = url
# COLUNA B = status (disponivel, sem estoque, produto indisponivel)
# COLUNA C = http_status
# COLUNA D = titulo
# COLUNA E = ultima_verificacao
# COLUNA F = observacao
# ==============================================================================

# --- INSTALAÇÃO DE DEPENDÊNCIAS NO AMBIENTE DO COLAB ---
import sys
if 'google.colab' in sys.modules:
    print("Instalando dependências do sistema e do Python (aguarde)...")
    # Nota: Removemos o oauth2client pois já não é necessário para a Opção 2
    !pip install -q playwright gspread pandas
    !playwright install chromium
    !playwright install-deps
    print("Instalação concluída com sucesso!\n")

import asyncio
import random
from datetime import datetime
import gspread
from google.colab import auth
from google.auth import default
from playwright.async_api import async_playwright

# ============================================
# CONFIGURAÇÕES
# ============================================
GOOGLE_SHEET_NAME = "Crawler Netshoes"
MAX_URLS = 1000
CONCURRENT_PAGES = 3  # Controla quantas abas abrem em simultâneo
HEADLESS = True

# ============================================
# AUTENTICAÇÃO GOOGLE SHEETS (NATIVA)
# ============================================
print("A iniciar autenticação nativa do Google Colab...")
# Isto vai abrir a janela pop-up para fazer login na sua conta Google
auth.authenticate_user()
creds, _ = default()

try:
    client = gspread.authorize(creds)
    sheet = client.open(GOOGLE_SHEET_NAME).sheet1
    print(f"Conectado com sucesso à planilha: '{GOOGLE_SHEET_NAME}'")
except gspread.exceptions.SpreadsheetNotFound:
    print(f"\nERRO: A planilha '{GOOGLE_SHEET_NAME}' não foi encontrada na sua conta Google.")
    print("Certifique-se de que o nome está exatamente igual (respeitando maiúsculas e minúsculas).")
    raise
except Exception as e:
    print(f"Erro inesperado na autenticação: {e}")
    raise e

# Ler todas as linhas da planilha
all_rows = sheet.get_all_values()
# Ignora o cabeçalho (linha 1) e limita ao MAX_URLS
rows = all_rows[1:MAX_URLS + 1]

# Semáforo para controlar a concorrência
semaphore = asyncio.Semaphore(CONCURRENT_PAGES)

# Dicionário na memória para guardar as respostas antes de salvar tudo em lote
resultados_memoria = {}

# ============================================
# DETECTAR STATUS DO PRODUTO (ATUALIZADO)
# ============================================
def detect_product_status(html: str, http_status: str):
    if http_status == "404":
        return "PRODUTO INDISPONIVEL", "Página retornou Erro 404 (Não encontrada)"

    html = html.lower()

    # Cenário específico solicitado: "acabou" deve retornar INDISPONIVEL
    if "acabou" in html:
        return "INDISPONIVEL", "Detectado sinal de produto que acabou"

    # Outros sinais de falta de estoque (retornam SEM ESTOQUE)
    out_of_stock_signals = ["esgotado", "avise-me", "produto indisponível", "indisponível"]

    # Sinais de produto ativo/disponível
    available_signals = ["adicionar ao carrinho", "comprar", "comprar agora"]

    # Verifica os outros sinais de falta de estoque
    for signal in out_of_stock_signals:
        if signal in html:
            return "SEM ESTOQUE", f"Detectado sinal de falta de estoque: '{signal}'"

    # Verifica disponibilidade
    for signal in available_signals:
        if signal in html:
            return "DISPONIVEL", "Produto disponível para compra"

    return "VERIFICAR", "Não foi possível determinar o status com os sinais atuais"

# ============================================
# PROCESSAR CADA URL INDIVIDUALMENTE
# ============================================
async def process_url(browser, row_index, url):
    async with semaphore:
        # Cria um contexto com User-Agent comum para evitar bloqueios simples de bots
        context = await browser.new_context(
            user_agent="Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36"
        )
        page = await context.new_page()

        now = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        status = "ERRO"
        http_status = "N/A"
        title = ""
        obs = ""

        try:
            print(f"[Processando] Linha {row_index} -> {url}")

            response = await page.goto(
                url,
                timeout=60000,
                wait_until="domcontentloaded"
            )

            if response:
                http_status = str(response.status)
            else:
                http_status = "Sem Resposta"

            # Pequena pausa aleatória para simular comportamento humano
            await asyncio.sleep(random.uniform(1.5, 3.5))

            html = await page.content()
            title = await page.title()
            title = title.strip() if title else ""

            # Validação do status
            status, obs = detect_product_status(html, http_status)
            print(f"[OK] Linha {row_index} finalizada -> {status}")

        except Exception as e:
            status = "ERRO"
            obs = str(e)[:150]
            print(f"[ERRO] Falha ao acessar Linha {row_index}: {obs}")

        finally:
            await page.close()
            await context.close()

            # Guarda o resultado na memória associado ao número da linha
            resultados_memoria[row_index] = [
                status,
                http_status,
                title,
                now,
                obs
            ]

# ============================================
# FUNÇÃO PRINCIPAL (MAIN)
# ============================================
async def main():
    print(f"\nIniciando extração de {len(rows)} URLs carregadas...")

    async with async_playwright() as p:
        browser = await p.chromium.launch(headless=HEADLESS)
        tasks = []

        # Criar a fila de tarefas assíncronas
        for idx, row in enumerate(rows, start=2): # start=2 porque a linha 1 é o cabeçalho
            if not row or len(row) == 0:
                continue

            url = row[0].strip()
            if not url or not url.startswith("http"):
                continue

            tasks.append(process_url(browser, idx, url))

        # Executa todas as URLs respeitando o semáforo de concorrência
        await asyncio.gather(*tasks)
        await browser.close()

    # -------------------------------------------------------------------------
    # ATUALIZAÇÃO EM LOTE (BATCH UPDATE) - EVITA BLOQUEIOS DA API DO GOOGLE
    # -------------------------------------------------------------------------
    print("\n[Gravação] Enviando todos os resultados para o Google Sheets de uma só vez...")

    if resultados_memoria:
        # Descobre qual foi a última linha processada para definir o tamanho da tabela
        max_idx = max(resultados_memoria.keys())

        data_to_update = []

        # Monta a matriz de dados para as colunas B, C, D, E e F
        for r_idx in range(2, max_idx + 1):
            if r_idx in resultados_memoria:
                data_to_update.append(resultados_memoria[r_idx])
            else:
                # Se alguma linha foi pulada, envia colunas vazias para manter o alinhamento
                data_to_update.append(["", "", "", "", ""])

        range_string = f"B2:F{max_idx}"

        try:
            # Faz um único "update" salvando todas as linhas de uma vez
            sheet.update(range_string, data_to_update)
            print(f"Sucesso! Planilha atualizada no intervalo {range_string} ({len(data_to_update)} linhas).")
        except Exception as batch_error:
            print(f"Erro ao salvar em lote: {batch_error}")
            print("Tentando salvar linha por linha como alternativa de segurança...")
            for r_idx, row_values in resultados_memoria.items():
                try:
                    sheet.update(f"B{r_idx}:F{r_idx}", [row_values])
                    await asyncio.sleep(1)
                except:
                    pass
    else:
        print("Nenhum resultado foi gerado para atualizar.")

    print("\n--- Processo Finalizado com Sucesso ---")

# ============================================
# EXECUÇÃO DO SCRIPT NO COLAB
# ============================================
await main()

Instalando dependências do sistema e do Python (aguarde)...


ERROR:asyncio:Future exception was never retrieved
future: <Future finished exception=TargetClosedError('Target page, context or browser has been closed')>
Traceback (most recent call last):
  File "/tmp/ipykernel_2328/2137142183.py", line 128, in process_url
    response = await page.goto(
               ^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/playwright/async_api/_generated.py", line 9611, in goto
    await self._impl_obj.goto(
  File "/usr/local/lib/python3.12/dist-packages/playwright/_impl/_page.py", line 557, in goto
    return await self._main_frame.goto(**locals_to_params(locals()))
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/playwright/_impl/_frame.py", line 156, in goto
    await self._channel.send(
  File "/usr/local/lib/python3.12/dist-packages/playwright/_impl/_connection.py", line 69, in send
    return await self._connection.wrap_api_call(
           ^^^^^^^^^^^^^^^^^^^^^^^^^

Installing dependencies...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:2 http://security.ubuntu.com/ubuntu jammy-security InRelease
Hit:3 https://cli.github.com/packages stable InRelease
Hit:4 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Hit:5 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Reading package lists... Done
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
fonts-freefont-ttf is already the newest version (20120503-10build1

/tmp/ipykernel_2328/1377734836.py:214: DeprecationWarning: The order of arguments in worksheet.update() has changed. Please pass values first and range_name secondor used named arguments (range_name=, values=)
  sheet.update(range_string, data_to_update)


Sucesso! Planilha atualizada no intervalo B2:F700 (699 linhas).

--- Processo Finalizado com Sucesso ---
